# Random Guide-Selection Strategies

Compares random strategies for selecting guide sets.

`driver.py` writes one `results.parquet` per run (a run == one strategy/config;
see `config.json`). `helpers.load_runs` stacks the runs listed in `RUNS` into a
single tidy long DataFrame — one row per leg, tagged with its display `mode` and
carrying `r_<metric>` = metric / that seed's unguided baseline. Every plot is an
Altair spec over that frame (see `plots.py`), so the cells below are a handful of
declarative calls.

In [ ]:
import altair as alt
import polars as pl

import helpers as H
import plots as P

# Recessive grids, top legends, the validated categorical palette (see plots.THEME).
alt.theme.register("analysis", enable=True)(lambda: P.THEME)

# Datasets here comfortably exceed Altair's 5k-row default guard; we render inline.
alt.data_transformers.enable("default", max_rows=None)


def ratio_long(df: pl.DataFrame, metrics: list[str]) -> pl.DataFrame:
    """Melt reached legs' `r_<metric>` columns into tidy (metric, ratio) rows."""
    return (
        df.filter(pl.col("reached"))
        .unpivot(
            index=["mode", "seed", "goal", "k"],
            on=[f"r_{m}" for m in metrics],
            variable_name="metric",
            value_name="ratio",
        )
        .with_columns(pl.col("metric").str.strip_prefix("r_"))
    )

In [ ]:
# Configuration: runs to overlay as (display_name, run-dir pattern). Strategy and
# k come from each run's config/data. Uncomment more to overlay them as modes.
RUNS = [
    ("500 MB", "run.1"),
    ("4 GB", "run.2"),
    # ("no-replacement-naive", ""),
    # ("smallest-overall", ""),
    # ("smallest-novel", ""),
]

# Cost metrics carried through the plots. `nodes_per_class` is available in the
# frame too; add it to any plot's metric list to surface it.
METRICS = ["iters", "nodes", "classes", "total_time"]

df, meta = H.load_runs(RUNS)
gr = H.goal_reach(df)

## Cost relative to per-seed baseline

Each reached leg is divided by its own seed's full unguided eqsat cost, so the
reference is a horizontal line at `ratio = 1.0` (guides break even; below 1 is
cheaper than unguided). Per k we show the median ratio with a 25th–75th
percentile band — ratios are heavy-tailed, so the median suits them better than a
mean. Per-seed pairing (not one flat average baseline) keeps a dip below the line
from reflecting which seeds are in the mix rather than whether guides helped.

In [ ]:
P.band_vs_k(
    ratio_long(df, METRICS),
    "ratio",
    meta,
    title="Cost relative to per-seed baseline",
    y_title="metric / seed baseline",
    metrics=METRICS,
    overlay_by="seed",  # faint per-seed median curves behind the band
    columns=2,
)

## Reachability summary

Three panels: leg-level reach rate vs k, goal coverage (fraction of goals with at
least one successful trial) vs k, and the CDF of per-goal reachability by k.

In [ ]:
P.reachability(df, gr, meta)

## Cost distribution by mode at each k

Box plots of reached-leg cost, one row per metric and one column per k. Node
count includes the guide egraph (`max(trial_nodes, guide_egraph_nodes)`).

In [ ]:
P.cost_boxplots(df, meta, ["iters", "nodes"])

## Summary table

In [ ]:
(
    df.group_by("mode", "k")
    .agg(
        pl.col("iters").mean().round(2).alias("avg_iters"),
        pl.col("nodes").mean().round(0).alias("avg_nodes"),
        pl.col("classes").mean().round(0).alias("avg_classes"),
        pl.col("nodes_per_class").mean().round(2).alias("avg_nodes_per_class"),
        pl.col("total_time").mean().round(4).alias("avg_time_s"),
        pl.col("reached").mean().round(3).alias("reached_rate"),
        pl.col("reached").sum().alias("n_reached"),
        pl.len().alias("n_legs"),
    )
    .sort("k", "mode")
)

## Goal-level reachability heatmap

Per-goal reach rate at each k, faceted by mode. Goals are sorted by average
reachability with the hardest at the top.

In [ ]:
P.reach_heatmap(gr, meta)

## Best-guide improvement over baseline (per goal)

For each goal at each k we take the best sampled guide — the minimum of
`metric / seed_baseline` over all guides that reached it — then summarise those
per-goal bests across goals with the median and a 25th–75th percentile band.
`ratio = 1.0` (dashed line) is break-even; below 1 means the best guide is cheaper
than unguided. Per-goal (not per-seed) and unitless.

In [ ]:
P.band_vs_k(
    ratio_long(df, ["nodes", "total_time"]),
    "ratio",
    meta,
    title="Best-guide improvement over baseline (per goal)",
    y_title="best metric / seed baseline",
    metrics=["nodes", "total_time"],
    group_by=["goal"],  # best (min ratio) guide per goal, then median over goals
    group_reduce="min",
    overlay_by="goal",  # faint per-goal best-ratio curves behind the band
)